# Lab 4 – Build and Evaluate a Mini RAG System

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/labs/04-rag_lab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

In this lab you will build a **Retrieval-Augmented Generation (RAG)** pipeline from scratch
and evaluate it rigorously using a **gold validation set** of questions with known answers.

You will:
- Split a mini-corpus into overlapping chunks
- Embed chunks and retrieve the most relevant ones for each query
- Measure **recall@k** to evaluate retrieval quality
- Measure **answer accuracy** using an **LLM-as-a-judge**
- Run experiments to see how **chunk size** affects both metrics

**Search for "TODO" lines to make sure you've answered all the questions!**

## Learning Goals

By completing this lab you will be able to:
1. Implement overlapping text chunking from scratch.
2. Retrieve relevant chunks using cosine similarity over embeddings.
3. Compute **recall@k** using a gold validation set with keyword labels.
4. Implement **LLM-as-a-judge** to measure answer accuracy against gold answers.
5. Run controlled experiments varying chunk size and analyze the trade-offs.

In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys; sys.path.append('./course-materials')
from course_utils import lab5_setup, lab5_generate_answer, get_sample_corpus, get_text_embedding

lab5_setup()

## Pre-Lab Questions

--- TODO

1. In a RAG system, why do we split documents into chunks instead of passing the entire corpus as context?
2. What is *recall@k* and why does it matter for retrieval quality?
3. What could go wrong if the LLM answers a question using information that is **not** in the retrieved context?

> Type your answers below each question.

--- TODO

## Scientific Question & Hypothesis

**Question:** How does chunk size affect retrieval recall and answer accuracy in a RAG system?

**Hypothesis:** I expect that ______________________________, because ________________________________.

**Experiment Plan:**
- Build a RAG pipeline over a small corpus with a gold validation set.
- Sweep over chunk sizes: 50, 150, and 300 characters.
- Measure recall@k and answer accuracy for each chunk size.
- Compare results against your hypothesis.

## Part 1 – Load and Explore the Corpus

We start with a small corpus of 5 documents on different topics.
Read through them — you’ll need to know what’s in them (and what’s **not**) to understand the evaluation later.

In [ ]:
corpus = get_sample_corpus('mini_wiki')
print(f'Loaded {len(corpus)} documents.\n')

for i, doc in enumerate(corpus):
    print(f'--- Document {i} ({len(doc)} chars) ---')
    print(doc)
    print()

## Part 2 – The Gold Validation Set

Below is a validation set of **(question, relevant_keywords, gold_answer)** tuples.
Each entry contains:

| Field | Purpose |
|---|---|
| `question` | A question to ask the RAG system |
| `relevant_keywords` | Keywords that *must* appear in retrieved context for recall @ k |
| `gold_answer` | The correct answer, used to score accuracy |
| `answerable` | `True` if the corpus contains the answer, `False` if not |

**Important:** The last two questions are **hallucination traps** — the facts needed to answer them
are *not* in the corpus. A well-grounded RAG system should respond “I don’t know,”
but LLMs often hallucinate from their training data instead.

In [ ]:
VALIDATION_SET = [
    # ---- Answerable, but HARDER: require combining facts / specific details ----
    {
        "question": "How many years did it take to construct the Eiffel Tower?",
        "relevant_keywords": ["1887", "1889"],
        "gold_answer": "About 2 years (constructed from 1887 to 1889).",
        "answerable": True,
    },
    {
        "question": "Was the Eiffel Tower well-received when it was first built?",
        "relevant_keywords": ["criticized", "artists"],
        "gold_answer": "No, it was initially criticized by some of France's leading artists.",
        "answerable": True,
    },
    {
        "question": "How many pyramids are in the Giza pyramid complex?",
        "relevant_keywords": ["three pyramids"],
        "gold_answer": "There are three pyramids in the Giza pyramid complex.",
        "answerable": True,
    },
    {
        "question": "In which modern city is the Great Pyramid of Giza located?",
        "relevant_keywords": ["greater cairo"],
        "gold_answer": "It borders present-day Giza in Greater Cairo, Egypt.",
        "answerable": True,
    },
    {
        "question": "Name the two oceans that bound the Pacific Ocean to the north and south.",
        "relevant_keywords": ["arctic", "southern ocean"],
        "gold_answer": "The Arctic Ocean to the north and the Southern Ocean to the south.",
        "answerable": True,
    },
    {
        "question": "How many nations have territory in the Amazon rainforest region?",
        "relevant_keywords": ["nine"],
        "gold_answer": "Nine nations have territory in the Amazon region.",
        "answerable": True,
    },
    # ---- NOT answerable from corpus (hallucination traps) ----
    # The model KNOWS these answers from training data but the corpus
    # does NOT contain them. With the grounding instruction removed,
    # the model will confidently hallucinate.
    {
        "question": "How tall is the Eiffel Tower in meters?",
        "relevant_keywords": ["eiffel tower"],
        "gold_answer": "The provided context does not mention the height of the Eiffel Tower.",
        "answerable": False,
    },
    {
        "question": "What is the deepest point in the Pacific Ocean called?",
        "relevant_keywords": ["pacific ocean"],
        "gold_answer": "The provided context does not name the deepest point of the Pacific Ocean.",
        "answerable": False,
    },
    {
        "question": "What species is most endangered in the Amazon rainforest?",
        "relevant_keywords": ["amazon"],
        "gold_answer": "The provided context does not mention any specific endangered species.",
        "answerable": False,
    },
    {
        "question": "How many stone blocks were used to build the Great Pyramid of Giza?",
        "relevant_keywords": ["pyramid", "giza"],
        "gold_answer": "The provided context does not mention the number of stones in the Great Pyramid.",
        "answerable": False,
    },
]

n_answerable = sum(1 for v in VALIDATION_SET if v['answerable'])
print(f'Validation set: {len(VALIDATION_SET)} questions')
print(f'  {n_answerable} answerable (require specific details from corpus)')
print(f'  {len(VALIDATION_SET) - n_answerable} hallucination traps (answer NOT in corpus)')

### How to Use the Validation Set

The validation set serves two purposes:

**1. Evaluating Retrieval (recall@k):**
For each question, retrieve the top-k chunks, then check how many of the
`relevant_keywords` appear in the retrieved text.
For example, the question "How many years did it take to construct the Eiffel Tower?"
has keywords `["1887", "1889"]`. If the retrieval finds chunks containing both dates,
recall is 1.0. If it only finds one date, recall is 0.5.
With a very small chunk size, dates and details often end up in different chunks, so
recall drops.

**2. Evaluating Answers (accuracy):**
After the LLM generates an answer from the retrieved context, compare it to the `gold_answer`
using an LLM judge.

There are two types of questions:
- **Answerable questions** require specific details (dates, counts, names) that may be
  split across chunks at small chunk sizes. The model may give incomplete or wrong answers
  when key details are missing from the retrieved context.
- **Hallucination traps** ask about facts the corpus never mentions (e.g., the height of
  the Eiffel Tower, endangered Amazon species). The gold answer says the information isn't
  available. If the LLM confidently provides a specific answer from its training data
  (e.g., "330 meters"), that's a hallucination and should be scored as **incorrect**.

## Part 3 – Implement Text Chunking

Implement a function that splits text into overlapping chunks of `chunk_size` **characters**,
with `overlap` characters shared between consecutive chunks.

**Example:** For `chunk_size=10`, `overlap=3`, and text `"abcdefghijklmno"`:
- Chunk 0: `"abcdefghij"` (chars 0–9)
- Chunk 1: `"hijklmno"` (chars 7–14, sharing 3 chars with previous)

In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):
    '''Split text into overlapping chunks of chunk_size characters.'''
    chunks = []
    # --- TODO: your code here ---
    return chunks

In [ ]:
# Test your chunking function on the first document
test_chunks = chunk_text(corpus[0], chunk_size=100, overlap=20)
print(f'Document 0 ({len(corpus[0])} chars) -> {len(test_chunks)} chunks of <= 100 chars\n')
for i, c in enumerate(test_chunks):
    print(f'  Chunk {i} ({len(c)} chars): "{c[:70]}..."')

## Part 4 – Embed and Retrieve

Implement two functions:

1. **`embed_corpus(chunks)`** — embed each chunk using `get_text_embedding()` and return a list of embedding vectors.
2. **`retrieve_top_k(query, chunks, chunk_embeddings, k)`** — find the k chunks most similar to the query using cosine similarity.

**Hint:** `get_text_embedding()` returns normalized vectors, so cosine similarity
is just the dot product: `np.dot(a, b)`.

In [ ]:
import numpy as np

def embed_corpus(chunks):
    '''Return a list of embedding vectors, one per chunk.'''
    # --- TODO: your code here ---
    pass

def retrieve_top_k(query, chunks, chunk_embeddings, k=3):
    '''Return the text of the top-k most similar chunks to the query.'''
    # --- TODO: your code here ---
    pass

In [ ]:
# Build the chunk index over the full corpus
all_text = '\n\n'.join(corpus)
chunks = chunk_text(all_text, chunk_size=150, overlap=30)
chunk_embeddings = embed_corpus(chunks)

print(f'Full corpus: {len(all_text)} chars -> {len(chunks)} chunks\n')

# Quick test: retrieve chunks for a sample query
query = 'Where is the Eiffel Tower?'
retrieved = retrieve_top_k(query, chunks, chunk_embeddings, k=3)
print(f'Query: "{query}"\n')
for i, c in enumerate(retrieved):
    print(f'  Retrieved {i}: "{c[:80]}..."')

## Part 5 – Evaluate Retrieval with Recall@k

Implement `compute_recall_at_k`: given a list of relevant keywords and the
retrieved chunks, return the **fraction of keywords found** in the retrieved text.

$$\text{recall@k} = \frac{\text{number of relevant keywords found in retrieved chunks}}{\text{total number of relevant keywords}}$$

**Example:**
- `relevant_keywords = ["champ de mars", "paris"]`
- Retrieved chunks contain `"paris"` but not `"champ de mars"`
- recall@k = 1/2 = **0.5**

**Note:** Do case-insensitive matching. Join all retrieved chunks into one string,
lowercase everything, then check if each keyword appears.

In [ ]:
def compute_recall_at_k(relevant_keywords, retrieved_chunks):
    '''Fraction of relevant keywords found in the retrieved chunks (case-insensitive).'''
    # solution provided:
    if not relevant_keywords:
        return 1.0
    # Join all retrieved chunks into one big string and lowercase
    retrieved_text = ' '.join(retrieved_chunks).lower()
    # Count how many keywords appear in the retrieved text
    found = sum(1 for kw in relevant_keywords if kw.lower() in retrieved_text)
    return found / len(relevant_keywords)

In [ ]:
# Test recall on the validation set with chunk_size=150
all_text = '\n\n'.join(corpus)
chunks = chunk_text(all_text, chunk_size=150, overlap=30)
chunk_embeddings = embed_corpus(chunks)

print(f'Testing recall@3 with {len(chunks)} chunks (chunk_size=150):\n')
for item in VALIDATION_SET:
    retrieved = retrieve_top_k(item['question'], chunks, chunk_embeddings, k=3)
    recall = compute_recall_at_k(item['relevant_keywords'], retrieved)
    status = '\u2705' if recall == 1.0 else '\u26a0\ufe0f'
    print(f'  {status} recall={recall:.2f}  "{item["question"]}"')

## Part 6 – Generate Answers and Evaluate Accuracy

### Generating Answers
Use `lab5_generate_answer(query, context)` to generate an answer from the retrieved chunks.
The LLM is prompted to answer **only** from the provided context and say “I don’t know” if unsure.

### LLM-as-a-Judge
To measure accuracy, we use an **LLM-as-a-Judge** approach:
instead of brittle exact-string matching, we ask an LLM to compare the generated answer
with the gold answer and decide if they are semantically equivalent.

This is a standard evaluation technique in AI engineering — the same LLM helper
(`lab5_generate_answer`) can be repurposed as a judge by passing the two answers
as “context” and asking a comparison question.

In [ ]:
def answer_accuracy(generated_answer, gold_answer):
    '''Use an LLM to judge if generated_answer matches gold_answer.

    Returns 1.0 if correct, 0.0 if incorrect.

    Approach: use lab5_generate_answer() as an LLM judge.
    Pass the two answers as "context" and ask the LLM to compare them.
    '''
    # solution provided:
    judge_query = (
        "Does Answer A convey the same core information as Answer B? "
        "If Answer B says the information is not available but Answer A "
        "provides a specific fact, that is INCORRECT (hallucination). "
        "Reply ONLY 'correct' or 'incorrect'."
    )
    judge_context = f"Answer A: {generated_answer}\nAnswer B: {gold_answer}"
    verdict = lab5_generate_answer(query=judge_query, context=judge_context)
    return 1.0 if "incorrect" not in verdict.lower() and "correct" in verdict.lower() else 0.0

In [ ]:
# Test: generate an answer and check accuracy for one example
item = VALIDATION_SET[0]
retrieved = retrieve_top_k(item['question'], chunks, chunk_embeddings, k=3)
context = '\n'.join(retrieved)
generated = lab5_generate_answer(query=item['question'], context=context)

print(f'Question:  {item["question"]}')
print(f'Generated: {generated}')
print(f'Gold:      {item["gold_answer"]}')
print(f'Accuracy:  {answer_accuracy(generated, item["gold_answer"])}')

In [ ]:
# Also test on a hallucination trap
trap = VALIDATION_SET[6]  # "How tall is the Eiffel Tower in meters?"
retrieved = retrieve_top_k(trap['question'], chunks, chunk_embeddings, k=3)
context = '\n'.join(retrieved)
generated = lab5_generate_answer(query=trap['question'], context=context)

print(f'Question:       {trap["question"]}')
print(f'Generated:      {generated}')
print(f'Gold:           {trap["gold_answer"]}')
print(f'Accuracy:       {answer_accuracy(generated, trap["gold_answer"])}')
print(f'\nDid the LLM hallucinate? Look at the generated answer \u2014')
print(f'the corpus never mentions the height of the Eiffel Tower!')

## Part 7 – Experiment: Chunk Size Sweep

Now run the full experiment. For each chunk size:
1. Chunk the entire corpus and build the embedding index.
2. For every validation question, retrieve top-k chunks.
3. Compute recall@k and answer accuracy.
4. Record mean scores.

**Why this works:** With small chunks (25 chars), key facts get fragmented
across chunks (e.g., “1887 to 1889” ends up in a different chunk than “Eiffel Tower”),
so retrieval misses them. With large chunks (150 chars), each chunk covers a full
document, so retrieval is much more likely to find the right context.

In [ ]:
import pandas as pd

chunk_sizes = [50, 150, 300]
k = 3
results = []
all_text = '\n\n'.join(corpus)

for size in chunk_sizes:
    print(f'\n{"="*60}')
    overlap = max(10, size // 5)
    chunks = chunk_text(all_text, chunk_size=size, overlap=overlap)
    chunk_embeddings = embed_corpus(chunks)
    print(f'Chunk size: {size}  |  Overlap: {overlap}  |  {len(chunks)} chunks')
    print(f'{"="*60}')

    recalls = []
    accuracies = []

    for item in VALIDATION_SET:
        retrieved = retrieve_top_k(item['question'], chunks, chunk_embeddings, k=k)
        context = '\n'.join(retrieved)

        # Recall
        recall = compute_recall_at_k(item['relevant_keywords'], retrieved)
        recalls.append(recall)

        # Generate answer & check accuracy
        generated = lab5_generate_answer(query=item['question'], context=context)
        acc = answer_accuracy(generated, item['gold_answer'])
        accuracies.append(acc)

        flag = '\u2705' if acc == 1.0 else '\u274c'
        print(f'  {flag} recall={recall:.2f} acc={acc:.0f}  {item["question"][:55]}')

    results.append({
        'chunk_size': size,
        'num_chunks': len(chunks),
        'mean_recall@k': round(np.mean(recalls), 3),
        'mean_accuracy': round(np.mean(accuracies), 3),
    })

df = pd.DataFrame(results)
print('\n')
df

## Part 8 – Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df['chunk_size'], df['mean_recall@k'], marker='o', linewidth=2, markersize=8, label='Mean Recall@k')
ax.plot(df['chunk_size'], df['mean_accuracy'], marker='s', linewidth=2, markersize=8, label='Mean Accuracy')
ax.set_xlabel('Chunk Size (characters)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Chunk Size vs. RAG Performance', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_xticks(chunk_sizes)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Results & Conclusion

--- TODO

Answer the following based on your experimental results:

1. Was your hypothesis supported? Why or why not?
2. Which chunk size gave the best recall? Best accuracy? Were they the same?
3. Did the hallucination traps catch the LLM? What did it generate for “How tall is the Eiffel Tower?”
   even though the height is nowhere in the corpus?
4. What trade-offs did you observe between small and large chunk sizes?

> Write your answers below.

## Post-Lab Reflection

--- TODO

1. Why is a gold validation set important for evaluating a RAG system?
2. What are the limitations of using “LLM-as-a-judge” for evaluation?
3. If you were building a RAG system for your project, what chunk size would you start with and why?

> Write your answers below.

---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot)
> you used while completing this lab or assignment.
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|-----------|---------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., “Why does my cosine similarity return NaN?”) | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**
Briefly describe what you learned or how AI helped you think through the problem.
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---

In [ ]:
# @title ✅ Run checks for Lab 5
assert callable(chunk_text), 'chunk_text must be defined'
assert callable(embed_corpus), 'embed_corpus must be defined'
assert callable(retrieve_top_k), 'retrieve_top_k must be defined'
assert callable(compute_recall_at_k), 'compute_recall_at_k must be defined'
assert callable(answer_accuracy), 'answer_accuracy must be defined'
print('\u2705 All required functions are defined!')

# Quick functional checks
test_chunks = chunk_text('Hello world this is a test sentence for chunking.', chunk_size=15, overlap=5)
assert len(test_chunks) > 0, 'chunk_text should return a non-empty list'
assert all(isinstance(c, str) for c in test_chunks), 'chunks should be strings'
assert all(len(c) <= 15 for c in test_chunks), 'no chunk should exceed chunk_size'

test_recall = compute_recall_at_k(['hello', 'world'], ['hello world this is a test'])
assert test_recall == 1.0, f'Expected recall 1.0, got {test_recall}'
test_recall2 = compute_recall_at_k(['hello', 'missing'], ['hello world'])
assert test_recall2 == 0.5, f'Expected recall 0.5, got {test_recall2}'

print('\u2705 Basic functional checks passed!')